In [2]:
import numpy as np
import itertools as it
from math import *
from enum import Enum

class TStatus(Enum):
    EVEN = 0
    VERTICAL = 1
    HORIZONTAL = 2

class Tensor:
    t: list
    dim: int
    covector: int
    vector: int
    deep: int
    status: TStatus
    
    def __init__(self, s, dim: int, covector: int, vector: int, status: TStatus = None, a_type=int):
        # Check dimensions and indexes
        if covector + vector <= 0:
            raise ValueError('ERR: 0 indexes for vectors or covectors')
            
        # Set dimensions
        self.dim = dim
        self.covector, self.vector = covector, vector
        self.deep = covector + vector
        
        # Check status
        if self.deep % 2:
            if status is not None and status != TStatus.EVEN:
                self.status = status
            else:
                self.status = (TStatus.VERTICAL if self.vector == 0 else TStatus.HORIZONTAL)
        else:
            self.status = TStatus.EVEN
        
        if isinstance(s, str):
            # Clear string data
            s = s.replace('−', '-').replace('\u200b', '').split()
            # Empty t
            self.t = np.zeros([self.dim] * self.deep, a_type)
            
            if self.status is TStatus.HORIZONTAL:
                divisors = [ind // 2 + (self.deep // 2 if ind % 2 else 0) for ind in range(self.deep - 1)] + [self.deep - 1]
            else:
                divisors = [ind // 2 + (self.deep // 2 + self.deep % 2 if ind % 2 else 0) for ind in range(self.deep)]
                
            for m in range(len(s)):
                index = [m // (self.dim ** div) % self.dim for div in divisors]
                self.t[*index] = s[m]
                # print(m + 1, '=', ', '.join(str(x + 1) for x in index), '=', s[m])
        elif isinstance(s, (np.ndarray, np.generic)):
            self.t = s
        else:
            self.t = np.zeros([self.dim] * self.deep, a_type)

    def _iterate_show(self):
        """Переборный интератор для show
        ИНДЕКСАЦИЯ: строка - столбец - суперстрока - суперстолбец - гиперстрока - гиперстолбец -
        """
        axes = list(range(self.deep - 1, -1, -1))
        if self.status is TStatus.HORIZONTAL:
            order = [ax for ax in axes if ax % 2 == 0 and ax != axes[0]] + [axes[0]] + [ax for ax in axes if ax % 2 != 0]
        else:
            order = [ax for ax in axes if ax % 2 == 0] + [ax for ax in axes if ax % 2 != 0]
        # print(axes, order)
        for indices in it.product(range(self.dim), repeat=self.deep):
            original_indices = [0] * self.deep
            for original_axis in range(self.deep):
                pos = order.index(original_axis)
                original_indices[original_axis] = indices[pos]
            yield tuple(original_indices)

    
    def show(self, sym_element=' ', sym_line='\n', sym_superline='\n\n', sym_superraw=' | ') -> str:
        """Превращает тензор в текст для вывода:
            sym_element: между двумя элементами, sym_line: перенос каждую строку,
            sym_superline: перенос каждую суперстроку, sym_superraw: перенос каждый суперстолбец.
        
        ИНДЕКСАЦИЯ: строка - столбец - суперстрока - суперстолбец - гиперстрока - гиперстолбец -
        """
        s = ''
        pre_line, pre_superline, pre_superraw = 0, 0, 0
        for indices in self._iterate_show():
            if len(indices) == 1:
                if (pre_line != indices[0]) and (self.status is TStatus.VERTICAL):
                    s += sym_line
                elif indices[0] != 0:
                    s += sym_element
                pre_line = indices[0]
            elif len(indices) == 2:
                if (pre_line != indices[0]):
                    s += sym_line
                elif indices[0] != 0 or indices[1] != 0:
                    s += sym_element
                pre_line = indices[0]
            elif len(indices) == 3:
                if self.status is TStatus.HORIZONTAL:
                    if (pre_line != indices[0]):
                        s += sym_line
                    elif (pre_superraw != indices[2]):
                        s += sym_superraw
                    elif indices[0] != 0 or indices[1] != 0:
                        s += sym_element
                    pre_line, pre_superraw = indices[0], indices[2]
                else:
                    if (pre_line != indices[0]):
                        s += sym_line
                    elif (pre_superline != indices[2]):
                        s += sym_superline
                    elif indices[0] != 0 or indices[1] != 0:
                        s += sym_element
                    pre_line, pre_superline = indices[0], indices[2]
            else:
                if (pre_line != indices[0]):
                    s += sym_line
                elif (pre_superline != indices[2]):
                        s += sym_superline
                elif (pre_superraw != indices[3]):
                    s += sym_superraw
                elif not all(i == 0 for i in indices):
                    s += sym_element
                pre_line, pre_superline, pre_superraw = indices[0], indices[2], indices[3]
            # print(indices)
            s += str(self.t[indices])
        return s
    
    @property
    def format(self):
        return '[' + self.show(', ', '; ', ', ', ', ') +']'
    
    def __getitem__(self, prop):
        return self.t[prop]
    
    def __setitem__(self, key, value):
        self.t[key] = value
    
    def __str__(self):
        return f'Tensor ({self.vector}, {self.covector}), dim = {self.dim}, status = {self.status}\n' + self.show() + '\n' + self.format
    
    def __repr__(self):
        return f'Tensor ({self.vector}, {self.covector})'
    
    def __mul__(self, other):
        return Tensor(other * self.t, self.dim, self.covector, self.vector, self.status)
    
    def __add__(self, other):
        return Tensor(self.t + other.t, self.dim, self.covector, self.vector, self.status)
    
    def __sub__(self, other):
        return Tensor(self.t - other.t, self.dim, self.covector, self.vector, self.status)
    
    @staticmethod
    def fuck_geolin(indices):
        indices = list(indices)
        if len(indices) <= 3:
            return indices
        start = 2  # Начало подсписка для обработки
        if len(indices) % 2:
            # Нечётная длина: обрабатываем элементы со 2 до предпоследнего
            sub_len = len(indices) - 3  # Длина подсписка indices[2:-1]
            processed = [indices[start + i + 1] if i % 2 == 0 else indices[start + i - 1] for i in range(sub_len)]
            return indices[:2] + processed + [indices[-1]]
        else:
            # Чётная длина: обрабатываем все элементы с индекса 2
            sub_len = len(indices) - start
            processed = [indices[start + i + 1] if i % 2 == 0 else indices[start + i - 1] for i in range(sub_len)]
            return indices[:2] + processed
    
    def __matmul__(A, B):
        dim = A.dim
        out = Tensor(0, dim, A.covector + B.covector, A.vector + B.vector, a_type=A.t.dtype)
        for COindicesA in it.product(range(dim), repeat=A.covector):
            for COindicesB in it.product(range(dim), repeat=B.covector):
                for indicesA in it.product(range(dim), repeat=A.vector):
                    for indicesB in it.product(range(dim), repeat=B.vector):
                        indices = Tensor.fuck_geolin(COindicesA + COindicesB + indicesA + indicesB)
                        out[*indices] = A[*(COindicesA+indicesA)] * B[*(COindicesB+indicesB)]     
        return out
        
tens = [
    Tensor('1 2', 2, 1, 0),
    Tensor('1 2', 2, 0, 1),
    Tensor('1 2 3 4', 2, 1, 1),
    Tensor('1 2 3 4 5 6 7 8', 2, 3, 0),
    Tensor('1 2 3 4 5 6 7 8', 2, 0, 3),
    Tensor('1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16', 2, 2, 2),
    Tensor(' '.join(str(x) for x in range(1, 32 + 1)), 2, 5, 0),
    Tensor(' '.join(str(x) for x in range(1, 32 + 1)), 2, 3, 2)
]
# [print('========== ', ten.status, '\n', ten, sep='') for ten in tens[:]]
    

### Задача 1

In [3]:
# i строка, j столбец, k суперстолбец, r суперстрока
data = """
−1 −1 0 1 −3 −1 4 0 2 ​ 4 1 4 4 3 1 −3 3 2 ​ 0 −2 1 −3 2 −4 3 −3 −1 ​ −1 −4 −3 1 −2 0 −4 2 4 ​ −1 3 0 4 −2 3 3 3 −4 ​ 3 3 0 −1 2 0 2 −3 4 ​ −2 −4 −1 2 4 3 3 2 0 ​ 1 0 0 −2 4 −1 −1 −2 −2 ​ 2 −3 −1 −4 1 −2 −1 −2 −3 ​ ​ ​"""
tensor = Tensor(data, 3, 2, 2)
print('A', tensor)
s = 0
for k in range(3):
    for r in range(3):
        # print(k + 1, r + 1, k + 1, r + 1, 'is', tensor[k][r][r][k])
        s += tensor[k][r][r][k]
        
print(f'[{s}]')

A Tensor (2, 2), dim = 3, status = TStatus.EVEN
-1 4 0 | -1 -1 3 | -2 1 2
-1 1 -2 | -4 3 3 | -4 0 -3
0 4 1 | -3 0 0 | -1 0 -1
1 4 -3 | 1 4 -1 | 2 -2 -4
-3 3 2 | -2 -2 2 | 4 4 1
-1 1 -4 | 0 3 0 | 3 -1 -2
4 -3 3 | -4 3 2 | 3 -1 -1
0 3 -3 | 2 3 -3 | 2 -2 -2
2 2 -1 | 4 -4 4 | 0 -2 -3
[-1, 4, 0, -1, -1, 3, -2, 1, 2; -1, 1, -2, -4, 3, 3, -4, 0, -3; 0, 4, 1, -3, 0, 0, -1, 0, -1; 1, 4, -3, 1, 4, -1, 2, -2, -4; -3, 3, 2, -2, -2, 2, 4, 4, 1; -1, 1, -4, 0, 3, 0, 3, -1, -2; 4, -3, 3, -4, 3, 2, 3, -1, -1; 0, 3, -3, 2, 3, -3, 2, -2, -2; 2, 2, -1, 4, -4, 4, 0, -2, -3]
[-8]


### Задача 2

In [4]:
# p строка, k столбец, t суперстолбец
tensor2A = Tensor('−3 4 ​ −3 3 ​ ​', 2, 0, 2)
# j строка, r столбец
tensor2B = Tensor('3 2 ​ −1 1 ​ −4 −2 ​ 1 3 ​ ​', 2, 2, 1)
print('A', tensor2A)
print('B', tensor2B)
res = tensor2A @ tensor2B
print('RES', res)

A Tensor (2, 0), dim = 2, status = TStatus.EVEN
-3 -3
4 3
[-3, -3; 4, 3]
B Tensor (1, 2), dim = 2, status = TStatus.HORIZONTAL
3 -1 | -4 1
2 1 | -2 3
[3, -1, -4, 1; 2, 1, -2, 3]
RES Tensor (3, 2), dim = 2, status = TStatus.HORIZONTAL
-9 3 | 12 -4 | 12 -3 | -16 4
-6 -3 | 8 4 | 6 -9 | -8 12
-9 3 | 9 -3 | 12 -3 | -12 3
-6 -3 | 6 3 | 6 -9 | -6 9
[-9, 3, 12, -4, 12, -3, -16, 4; -6, -3, 8, 4, 6, -9, -8, 12; -9, 3, 9, -3, 12, -3, -12, 3; -6, -3, 6, 3, 6, -9, -6, 9]


### Задача 3

In [5]:
data = '''
1 2 2 −3 ​ 3 2 0 4 ​ 3 −3 −2 0 ​ 4 0 2 0 ​ −1 1 3 −4 ​ 0 −3 −1 −1 ​ 3 1 −3 3 ​ −3 2 2 3 ​ ​ ​
'''
A = Tensor(data, 2, 4, 1)
print(A)
C = Tensor(0, 2, 3, 0, TStatus.HORIZONTAL)
for t in range(2):
    for l in range(2):
        for k in range(2):
            for p in range(2):
                # 
                C[t][k][p] += A[t][l][p][k][l]
                
print(C)

Tensor (1, 4), dim = 2, status = TStatus.HORIZONTAL
1 3 | 3 4 | -1 0 | 3 -3
2 2 | -3 0 | 1 -3 | 1 2
2 0 | -2 2 | 3 -1 | -3 2
-3 4 | 0 0 | -4 -1 | 3 3
[1, 3, 3, 4, -1, 0, 3, -3; 2, 2, -3, 0, 1, -3, 1, 2; 2, 0, -2, 2, 3, -1, -3, 2; -3, 4, 0, 0, -4, -1, 3, 3]
Tensor (0, 3), dim = 2, status = TStatus.HORIZONTAL
1 0 | 1 0
-1 -1 | -4 3
[1, 0, 1, 0; -1, -1, -4, 3]


### Задача 4

In [6]:
dataA = '6 −4 4 ​ −1 −2 −3 ​ 1 −4 1 ​ ​​'
dataB = '4 −5 −6 ​ ​'
dataC = '−2 −4 6 ​ 0 −5 −2 ​ −5 4 0 ​ ​'

A = Tensor(dataA, 3, 0, 2)
B = Tensor(dataB, 3, 1, 0)
C = Tensor(dataC, 3, 2, 0)
print(A, B, C, sep='\n')

D = Tensor(0, 3, 1, 0)
for s in range(3):
    for j in range(3):
        for p in range(3):
            D[s] += A[j][p] * B[j] * C[p][s]
print(D)

Tensor (2, 0), dim = 3, status = TStatus.EVEN
6 -1 1
-4 -2 -4
4 -3 1
[6, -1, 1; -4, -2, -4; 4, -3, 1]
Tensor (0, 1), dim = 3, status = TStatus.VERTICAL
4
-5
-6
[4; -5; -6]
Tensor (0, 2), dim = 3, status = TStatus.EVEN
-2 0 -5
-4 -5 4
6 -2 0
[-2, 0, -5; -4, -5, 4; 6, -2, 0]
Tensor (0, 1), dim = 3, status = TStatus.VERTICAL
-28
-156
-4
[-28; -156; -4]


### Задача 5

In [7]:
A = Tensor('−1 1 −3 −1 ​ −5 −3 1 −3 ​ −1 −4 5 −3 ​ −2 −2 2 −2 ​ ​', 4, 1, 1)
print(A)
v = (3, 4, -3, -3)
u = (-5, 2, -2, -5)
s = 0
for m in range(4):
    for p in range(4):
        s += A[m][p] * v[p] * u[m]
print(s)

Tensor (1, 1), dim = 4, status = TStatus.EVEN
-1 -5 -1 -2
1 -3 -4 -2
-3 1 5 2
-1 -3 -3 -2
[-1, -5, -1, -2; 1, -3, -4, -2; -3, 1, 5, 2; -1, -3, -3, -2]
140


### Задача 6

In [8]:
A = Tensor('0 1 1 ​ −4 −2 2 ​ −2 −2 0 ​ ​', 3, 0, 2)
print(A)
B = A * -3
print(B)

Tensor (2, 0), dim = 3, status = TStatus.EVEN
0 -4 -2
1 -2 -2
1 2 0
[0, -4, -2; 1, -2, -2; 1, 2, 0]
Tensor (2, 0), dim = 3, status = TStatus.EVEN
0 12 6
-3 6 6
-3 -6 0
[0, 12, 6; -3, 6, 6; -3, -6, 0]


### Задача 7
Местами не перепутай: ко=ЛФ, контра=обычные

### Задача 8

In [9]:
# a ⊗ (3b⊗c−4d⊗e)
A = Tensor('-1 3 3 -4', 2, 0, 2)
B = Tensor('-4 -4', 2, 0, 1)
C = Tensor('-1 3', 2, 1, 0)
D = 3
E = Tensor('-1 0 5 -2', 2, 1, 1)
# print(A, B, C, D, E, sep='\n===\n')

R = A @ (((B * 3) @ C) + (E * (D * -4)))

print(R.format)

[-24, 72, 72, -96; 36, -108, -108, 144; 48, -144, -144, 192; 12, -36, -36, 48]


In [10]:
# СЕМЁН

A = Tensor('1 3 -4 -3', 2, 0, 2)
B = Tensor('-3 -6', 2, 0, 1)
C = Tensor('3 2', 2, 1, 0)
D = Tensor('0 -6', 2, 1, 0)
E = Tensor('3 -3', 2, 0, 1)
# print(A, B, C, D, E, sep='\n===\n')

R = A @ (((B * -5) @ C) + ((D * 3) @ E))
print(R.format)

[45, 135, -180, -135; -24, -72, 96, 72; 90, 270, -360, -270; 114, 342, -456, -342]


### Задача 9

In [11]:
data = '''
4 −8 −7 4 8 −6 7 7 2 ​ −1 3 7 −5 7 −1 −3 4 −8 ​ 2 −5 −5 −7 1 9 −2 −9 5 ​ −7 −3 1 9 4 2 −4 5 8 ​ 6 −6 −6 0 2 5 −7 −1 8 ​ 5 −5 5 8 −2 −9 0 0 2 ​ 0 −2 1 −5 1 8 −8 −2 6 ​ −6 0 −7 −9 7 −7 −2 4 4 ​ −2 −8 3 −2 5 6 7 −7 3 ​ −7 0 3 2 4 −8 −1 5 7 ​ −7 5 −2 −3 −5 −1 −1 −7 −3 ​ 7 1 −3 −3 −6 8 7 −7 −3 ​ 5 8 5 −8 6 −9 3 8 −9 ​ 6 2 4 −2 8 5 −5 6 −1 ​ 5 5 4 3 8 9 8 4 3 ​ 6 7 2 −3 −2 4 9 6 −8 ​ −9 −7 −4 2 0 4 −5 −1 −6 ​ 1 7 5 −1 −2 3 −5 0 −4 ​ −9 −1 7 −8 2 −2 −6 −8 2 ​ −1 −8 −5 8 5 3 −1 −4 5 ​ −5 −1 6 3 −4 5 −4 6 8 ​ 7 4 0 −5 5 2 5 −1 4 ​ 4 −4 −5 −3 8 7 8 0 −7 ​ 4 2 3 7 −4 −7 3 −5 2 ​ 2 −2 −6 8 2 0 −5 −8 6 ​ −1 4 9 5 −8 4 −6 1 −1 ​ −8 1 −8 0 −6 1 −3 0 1 ​ ​ ​
'''
A = Tensor(data, 3, 4, 1)
# print(A)
k, j, n, p, l = 2, 2, 2, 1, 1
print(A[k - 1][j - 1][p - 1][n - 1][l - 1])

-6


### Задача 10

In [15]:
data = "4 1 −8 ​ 3 2 −1 ​ 8 3 7 ​ −3 8 2 ​ 2 0 3 ​ 8 −1 4 ​ 7 −3 −1 ​ 6 3 3 ​ 7 −4 3 ​ ​​"
a = Tensor(data, 3, 0, 3)
print(a)

e1, e2, e3 = [1, -2, -2], [-1, 3, 4], [2, -5, -5]
old_basis = np.matrix([[e1[0], e2[0], e3[0]], [e1[1], e2[1], e3[1]], [e1[2], e2[2], e3[2]]], float)
e1, e2, e3 = [1, -2, 3], [1, -1, 1], [2, -6, 11]
new_basis = np.matrix([[e1[0], e2[0], e3[0]], [e1[1], e2[1], e3[1]], [e1[2], e2[2], e3[2]]], float)
# print(old_basis, new_basis, sep='\n')

S = np.linalg.inv(new_basis).dot(old_basis)
T = np.around(np.linalg.inv(S), 12)
T, S = np.asarray(T), np.asarray(S)
print(T, S, sep='\n')

tilde_a = np.zeros((3, 3, 3))
for m in range(3):
    for p in range(3):
        for l in range(3):
            total = 0
            for i1 in range(3):
                for i2 in range(3):
                    for i3 in range(3):
                        total += T[i1, m] * T[i2, p] * T[i3, l] * a[i1, i2, i3]
            tilde_a[m, p, l] = total

tilde_a = np.round(tilde_a, 10).astype(int)
res_tensor = Tensor(tilde_a, 3, 2, 1)
print(res_tensor)

Tensor (3, 0), dim = 3, status = TStatus.HORIZONTAL
4 3 8 | -3 2 8 | 7 6 7
1 2 3 | 8 0 -1 | -3 3 -4
-8 -1 7 | 2 3 4 | -1 3 3
[4, 3, 8, -3, 2, 8, 7, 6, 7; 1, 2, 3, 8, 0, -1, -3, 3, -4; -8, -1, 7, 2, 3, 4, -1, 3, 3]
[[ -4.   1. -19.]
 [  5.   2.  17.]
 [  5.   1.  19.]]
[[ 21. -38.  55.]
 [-10.  19. -27.]
 [ -5.   9. -13.]]
Tensor (1, 2), dim = 3, status = TStatus.HORIZONTAL
-2241 138 -9743 | -280 2 -1185 | -8904 585 -38789
-72 211 -741 | 258 133 812 | -847 614 -4844
-9336 137 -39682 | -1732 -270 -6744 | -35906 1168 -153957
[-2241, 138, -9743, -280, 2, -1185, -8904, 585, -38789; -72, 211, -741, 258, 133, 812, -847, 614, -4844; -9336, 137, -39682, -1732, -270, -6744, -35906, 1168, -153957]


In [13]:
# new_a = np.einsum('ia,bj,ck,abc->ijk', S, T, T, A.t)
# new_a = np.around(new_a, 10)
# print(Tensor(new_a, 3, 2, 1))
# print(T, S, sep='\n')
# A_new = Tensor(0, 3, 2, 1)
# for t, k, n, j1, j2, i1 in it.product(range(3), repeat=6):
#     A_new[t][k][n] += round(T.item(i1, n) * S.item(t, j1) * S.item(k, j2) * A[j1][j2][i1], 12)
#     # if t==0 and k==0 and n==0:
#     #     print(t, k, n, j1, j2, i1, '=', round(T.item(i1, n), 12), S.item(t, j1), S.item(k, j2), A[j1][j2][i1], '=', round(T.item(i1, n) * S.item(t, j1) * S.item(k, j2) * A[j1][j2][i1], 12), A_new[t][k][n])
# print(A_new)

In [14]:
import numpy as np

# Инициализация тензора A согласно условию (j-строка, m-столбец, t-слой)
A = np.array([
    [[2, 4, -4], [-5, 5, -3], [-8, 7, 1]],    # j=0 (первая строка матрицы A)
    [[8, 5, -1], [-5, 5, 7], [-6, 4, -2]],    # j=1 (вторая строка)
    [[-7, -3, -6], [-2, 6, -2], [-3, -7, -1]] # j=2 (третья строка)
], dtype=int)

# Старый базис (e1, e2, e3)
old_basis = np.array([
    [-1, 1, 0],   # e1
    [2, -1, -1],  # e2
    [1, -2, 2]    # e3
]).T

# Новый базис (tilde_e1, tilde_e2, tilde_e3)
new_basis = np.array([
    [1, 2, 1],    # tilde_e1
    [1, 3, 2],    # tilde_e2
    [-2, -6, -3]  # tilde_e3
]).T

# Матрица перехода S: old_basis = new_basis * S
S = np.linalg.inv(new_basis) @ old_basis
T = np.linalg.inv(S)  # Обратное преобразование

# Преобразование тензора: tilde_A^n_{tk} = T^t_p T^k_q S^r_n A^r_{pq}
tilde_A = np.einsum('tp,kq,rn,rpq->ntk', T, T, S, A).astype(int)

print("Правильная матрица тензора в новом базисе:")
print(Tensor(tilde_A, 3, 2, 1))

Правильная матрица тензора в новом базисе:
Tensor (1, 2), dim = 3, status = TStatus.HORIZONTAL
-65505 -26616 -15496 | -26621 -10817 -6296 | -15410 -6261 -3646
109256 44228 25910 | 44390 17970 10525 | 25588 10358 6069
89792 36773 21131 | 36518 14956 8591 | 21317 8728 5018
[-65505, -26616, -15496, -26621, -10817, -6296, -15410, -6261, -3646; 109256, 44228, 25910, 44390, 17970, 10525, 25588, 10358, 6069; 89792, 36773, 21131, 36518, 14956, 8591, 21317, 8728, 5018]
